## Importações

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

# years = [2019, 2020, 2021, 2022, 2023, 2024]
years = range(2019,2025)

# modes = {}
# graphs = {}

# base_modes = "../../data/03_modes"

# for year in years:
#     for speed in ["fast", "medium", "slow"]:
#         file_path = f"../../data/04_graphs/{year}/{speed}_graph.gpickle"
#         with open(file_path, "rb") as f:
#             graphs[f"graph_{speed}_{year}"] = pickle.load(f)
                    
#             modes[f"{speed}_{year}"] = pd.read_parquet(f"../../data/03_modes/{year}/{speed}_emd.parquet")

graphs = {}
returns = {}

base_modes = "../../data/02_clean"

for year in years:
    file_path = f"../../data/04_graphs/{year}/graph.gpickle"
    with open(file_path, "rb") as f:
        graphs[f"graph_{year}"] = pickle.load(f)
                
        returns[year] = pd.read_parquet(f"../../data/02_clean/returns_{year}.parquet")
        returns[year] = np.log(1 + returns[year])

## Funções

In [88]:
import community as community_louvain

def calculate_hybrid_risk_metric(G, weights=None):
    # Default equal weights if not provided
    if weights is None:
        weights = {
            'degree': 1/4,
            'closeness': 1/4,
            'betweenness': 1/4,
            'eigenvector': 1/4
        }
    
    nodes = list(G.nodes)

    G = G.copy()
    for u, v, d in G.edges(data=True):
        d["distance"] = 1 / abs(d["weight"]) if d["weight"] != 0 else np.inf
    
    # 1. Calculate raw metrics
    
    # Degree
    degree_dict = dict(nx.degree(G, weight="weight"))
    
    # Clustering coefficient
    closeness_dict = nx.closeness_centrality(G, distance="distance")
    
    # Betweenness centrality
    betweenness_dict = nx.betweenness_centrality(G, weight='distance', normalized=True)
    
    # Eigenvector centrality
    eig_dict = nx.eigenvector_centrality(G, weight="weight")
    
    # 2. Normalize metrics (0=lowest risk, 1=highest risk)
    def normalize_dict(d, invert=False):
        vals = np.array(list(d.values()))
        min_val = vals.min()
        max_val = vals.max()
        if max_val - min_val == 0:
            # Avoid division by zero if all values equal
            return {k: 0.0 for k in d.keys()}
        norm = {k: (v - min_val) / (max_val - min_val) for k, v in d.items()}
        if invert:
            norm = {k: 1 - v for k, v in norm.items()}
        return norm
    
    norm_degree = normalize_dict(degree_dict, invert=False)
    norm_closeness = normalize_dict(closeness_dict, invert=False)
    norm_betweenness = normalize_dict(betweenness_dict, invert=False)
    norm_eig = normalize_dict(eig_dict, invert=False)
    
    # 3. Combine metrics with weights
    risk_scores = {}
    for node in nodes:
        score = (
            weights['degree'] * norm_degree[node] +
            weights['closeness'] * norm_closeness[node] +
            weights['betweenness'] * norm_betweenness[node] +
            weights['eigenvector'] * norm_eig[node] 
        )
        risk_scores[node] = score
    
    return risk_scores

## Construção do DataFrame com os nós e as métricas de grafos calculadas para cada modo

In [89]:
for name, G in graphs.items():
    print(name)

graph_2019
graph_2020
graph_2021
graph_2022
graph_2023
graph_2024


In [90]:
data = []

for name, G in graphs.items():
    _, year = name.split("_")

    hrm = calculate_hybrid_risk_metric(G)

    for node in G.nodes():
        data.append({
            "node": node,
            "hrm": hrm[node],
            "year": year
        })

df = pd.DataFrame(data)

In [92]:
central_portfolios = {}
peripheral_portfolios = {}

q75 = df.groupby("year")["hrm"].quantile(0.80).reset_index(name="p75")
q25 = df.groupby("year")["hrm"].quantile(0.20).reset_index(name="p25")

for year in years:
    p75 = q75.loc[q75["year"] == str(year), "p75"].iloc[0]
    p25 = q25.loc[q25["year"] == str(year), "p25"].iloc[0]

    central_portfolios[year] = df[(df["year"] == str(year)) & (df["hrm"] > p75)]
    peripheral_portfolios[year] = df[(df["year"] == str(year)) & (df["hrm"] < p25)]

In [93]:
centralities = ["central", "peripheral"]

def get_portfolio_info(
        portfolio
):
    stock_info = pd.read_csv("../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv")
    screener = pd.read_parquet("../../data/01_raw/screener_result.parquet")[["Ticker", "Country"]]
    stock_info = stock_info.merge(
        screener,
        how="left",
        on="Ticker"
    )
    portfolio = portfolio.to_frame().rename(columns={"node":"Ticker"})
    return portfolio.merge(
        stock_info, on="Ticker", how="left"
    )

for year in years:
    central_temp = central_portfolios[year]["node"]
    peripheral_temp = peripheral_portfolios[year]["node"]
    returns_temp = pd.read_parquet(
        f"../../data/02_clean/returns_{year}.parquet"
    )
    central_port_temp = returns_temp[central_temp]
    peripheral_port_temp = returns_temp[peripheral_temp]

    central_metadata_temp = get_portfolio_info(central_temp)
    peripheral_metadata_temp = get_portfolio_info(peripheral_temp)

    central_port_temp.to_csv(f"../../data/06_portfolios/central_{year}.csv")
    peripheral_port_temp.to_csv(f"../../data/06_portfolios/peripheral_{year}.csv")

    central_metadata_temp.to_csv(f"../../data/07_portfolios_metadata/central_metadata_{year}.csv")
    peripheral_metadata_temp.to_csv(f"../../data/07_portfolios_metadata/peripheral_metadata_{year}.csv")